# Case study: catching Telco churners when churners are the minority

*Session 8, block 2 worked example. Author: course team, licence CC-BY-4.0.*

**The question.** 26.5 % of the IBM Telco customers left. A model trained on such data predicts "stays" most of the time and misses many churners. The retention team wants to find more of them. Is it better to change the *training data* (undersampling, oversampling, SMOTE), the *loss* (class weights), or only the *decision* (a threshold chosen from the costs)?

- **Part A.** Strategies inside imbalanced-learn pipelines (undersampling, oversampling, SMOTENC, class weights) with a logistic regression and a random forest, compared with a lower threshold. Measures: precision, recall, F1 of the churn class, ROC AUC and PR AUC.
- **Part B.** The same strategies judged by the cost of a retention campaign (offer €100, lost customer €400, as in [06-case-study-churn-threshold-and-calibration.ipynb](06-case-study-churn-threshold-and-calibration.ipynb)), against a threshold tuned by `TunedThresholdClassifierCV`.

Theory: [04-lecture-scores-thresholds-and-rare-classes.ipynb](04-lecture-scores-thresholds-and-rare-classes.ipynb). The notebook downloads the Telco data from GitHub and runs in about two minutes.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, make_scorer,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Part A: which customers will leave?

In [2]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print(y.value_counts().to_dict(), "churn rate", round(y.mean(), 3))
cv = StratifiedKFold(5, shuffle=True, random_state=0)

{0: 5174, 1: 1869} churn rate 0.265


SMOTENC needs to know which columns are categorical *after* preprocessing. We therefore encode the categories as integers (ordinal) and scale the numbers first, let SMOTENC interpolate only the numeric columns, and one-hot encode afterwards for the logistic regression. The forest can work with the ordinal codes directly.

In [3]:
ordinal = make_column_transformer((OrdinalEncoder(), cat_cols), remainder=StandardScaler())   # categories first
cat_idx = list(range(len(cat_cols)))
onehot = make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_idx),
                                 remainder="passthrough")


def strategies(model, weighted):
    return {
        "no correction": make_pipeline(ordinal, onehot, model()),
        "undersampling": make_pipeline(ordinal, RandomUnderSampler(random_state=0), onehot, model()),
        "oversampling": make_pipeline(ordinal, RandomOverSampler(random_state=0), onehot, model()),
        "SMOTENC": make_pipeline(ordinal, SMOTENC(categorical_features=cat_idx, random_state=0), onehot, model()),
        "class weights": make_pipeline(ordinal, onehot, weighted()),
    }


models = {
    "logistic regression": strategies(lambda: LogisticRegression(max_iter=2000),
                                      lambda: LogisticRegression(max_iter=2000, class_weight="balanced")),
    "random forest": strategies(lambda: RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1),
                                lambda: RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced",
                                                               random_state=0, n_jobs=-1)),
}


def scores(proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"precision": precision_score(y, pred), "recall": recall_score(y, pred), "f1": f1_score(y, pred),
            "accuracy": accuracy_score(y, pred), "roc_auc": roc_auc_score(y, proba),
            "pr_auc": average_precision_score(y, proba)}


rows, probas = [], {}
for model_name, cands in models.items():
    for name, pipe in cands.items():
        proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]   # samplers: training folds only
        probas[(model_name, name)] = proba
        rows.append({"model": model_name, "strategy": name, **scores(proba)})
pd.DataFrame(rows).set_index(["model", "strategy"]).round(3)

precision  recall     f1  accuracy  roc_auc  pr_auc
model               strategy                                                          
logistic regression no correction      0.656   0.546  0.596     0.803    0.845   0.653
                    undersampling      0.516   0.805  0.628     0.748    0.844   0.644
                    oversampling       0.514   0.805  0.627     0.746    0.845   0.651
                    SMOTENC            0.529   0.776  0.629     0.757    0.841   0.644
                    class weights      0.516   0.803  0.628     0.748    0.845   0.651
random forest       no correction      0.669   0.503  0.574     0.802    0.844   0.654
                    undersampling      0.522   0.793  0.630     0.752    0.845   0.647
                    oversampling       0.549   0.731  0.627     0.770    0.841   0.644
                    SMOTENC            0.564   0.691  0.621     0.777    0.839   0.631
                    class weights      0.538   0.761  0.630     0.763    0.844   0.649

### The alternative: keep the model, move the threshold

A threshold chosen on the cross-validated probabilities of the uncorrected model. Here we pick the threshold with the best F1, for illustration; in practice choose it from costs (Part B) and on validation data only.

In [4]:
for model_name in models:
    proba = probas[(model_name, "no correction")]
    prec, rec, thr = precision_recall_curve(y, proba)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    best = thr[np.argmax(f1[:-1])]
    print(model_name, "best-F1 threshold", round(best, 2), {k: round(v, 3) for k, v in scores(proba, best).items()})

logistic regression best-F1 threshold 0.29 {'precision': 0.538, 'recall': 0.781, 'f1': 0.637, 'accuracy': 0.764, 'roc_auc': 0.845, 'pr_auc': 0.653}
random forest best-F1 threshold 0.26 {'precision': 0.513, 'recall': 0.821, 'f1': 0.632, 'accuracy': 0.746, 'roc_auc': 0.844, 'pr_auc': 0.654}


**Questions.**

1. Which strategies change ROC AUC and PR AUC, and which only move precision and recall? What does that tell you?
2. Compare the best correction with the uncorrected model at its best threshold. Is resampling worth the extra step?
3. A retention team can call 1,000 customers a month. Which number matters to them: F1, recall, or the precision among the 1,000 customers with the highest predicted risk? Compute the last one for two strategies.

In [5]:
# precision among the 1,000 customers with the highest predicted churn risk
for key in [("logistic regression", "no correction"), ("logistic regression", "SMOTENC")]:
    top = np.argsort(-probas[key])[:1000]
    print(key, "precision in top 1,000:", round(y.iloc[top].mean(), 3))

('logistic regression', 'no correction') precision in top 1,000: 0.71
('logistic regression', 'SMOTENC') precision in top 1,000: 0.707


# Part B: which strategy is cheapest for the retention team?

The threshold case study priced the decision: an offer costs €100, a churner who is not reached costs €400. We compute the campaign cost of every strategy at its default threshold 0.5 from the cross-validated probabilities of Part A, and compare with the uncorrected logistic regression whose threshold `TunedThresholdClassifierCV` chooses from these costs inside each training fold (so its predictions are as honest as the others).

In [6]:
OFFER, LOST = 100, 400


def campaign_cost(y_true, pred):
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    return OFFER * (pred == 1).sum() + LOST * ((y_true == 1) & (pred == 0)).sum()


rows = [{"model": m, "strategy": s, "offers": int((p >= 0.5).sum()), "cost €": campaign_cost(y, (p >= 0.5).astype(int))}
        for (m, s), p in probas.items()]
tuned = TunedThresholdClassifierCV(models["logistic regression"]["no correction"],
                                   scoring=make_scorer(lambda yt, yp: -campaign_cost(yt, yp)), cv=5, random_state=0)
pred_tuned = cross_val_predict(tuned, X, y, cv=cv)              # threshold chosen inside each training fold
rows.append({"model": "logistic regression", "strategy": "no correction, cost-tuned threshold",
             "offers": int(pred_tuned.sum()), "cost €": campaign_cost(y, pred_tuned)})
cost_table = pd.DataFrame(rows).sort_values("cost €")
cost_table["cost per customer €"] = (cost_table["cost €"] / len(y)).round(1)
cost_table

,model,strategy,offers,cost €,cost per customer €
1,logistic regression,undersampling,2917,437700,62.1
4,logistic regression,class weights,2909,438100,62.2
6,random forest,undersampling,2842,438600,62.3
2,logistic regression,oversampling,2928,438800,62.3
10,logistic regression,"no correction, cost-tuned threshold",2963,440700,62.6
3,logistic regression,SMOTENC,2743,441500,62.7
9,random forest,class weights,2642,443000,62.9
7,random forest,oversampling,2486,449800,63.9
8,random forest,SMOTENC,2289,459700,65.3
0,logistic regression,no correction,1557,494900,70.3


In our run the uncorrected models at threshold 0.5 cost about €70–73 per customer; every correction and the cost-tuned threshold cost €62–65, within about €1 of each other. With these costs the optimal threshold (c / L = 0.25) is close to the churn rate (0.265), and balancing the classes moves a calibrated model's threshold to roughly that point.

**Questions.** (1) Which strategy is cheapest? By how much does it beat the uncorrected model at threshold 0.5? (2) The resampling strategies and class weights all move the model towards "churn". Why do they land close to the cost-tuned threshold here, and why would they not if a lost customer were worth €1,000? (3) Which solution would you hand to the retention team, and which one keeps probabilities that Finance can sum up into an expected number of churners (lecture notebook 04, calibration)?